# Autodata — analiză statică reală cu REA (morluto/rea)

Acest notebook folosește **REA 6.8.0**, [proiectul oficial morluto/rea](https://github.com/morluto/rea), pentru a inspecta **executabilul original `ADCDA2/ADBCD.exe`** din arhiva proprie aflată în Google Drive.

**Pe telefon:** în Colab deschide **Runtime → Run all** și acordă acces la Google Drive. Notebookul descarcă doar `autodata_part_0001.zip`, extrage numai `ADBCD.exe`, verifică SHA256 și execută strict analize *statice* REA (fără lansarea executabilului Autodata).

**Rezultate:** folderul `Autodata_REA_Static_Reports` din folderul Drive Autodata, cu rapoartele `rea_pe_imports.json`, `rea_desktop_application.json` și `REA_SUMMARY.json`.

**Limitări:** rapoartele PE identifică importuri și structura executabilului. Nu decodifică automat fișierele de date `.IDF`, nu generează codul-sursă original și nu ocolesc o licență. Nu se rulează activatoare, drivere sau fișiere blocate de Google Drive.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import json, io, hashlib, zipfile, os, sys, subprocess, time, shutil
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaInMemoryUpload
from googleapiclient.errors import HttpError

ROOT_ID="1XqpDaYI5zkdUXY85wLSvpsklatT0P0Bd"
ARCHIVE_NAME="autodata_part_0001.zip"
EXE_MEMBER="Autodata__3.45/ADCDA2/ADBCD.exe"
EXE_SHA256="a505bbc4f8678713f352faa8fc70337d9edbe5e0ebd23ba305ca3bccd0f6fce4"
OUT_NAME="Autodata_REA_Static_Reports"
WORK=Path("/content/autodata_rea_static"); WORK.mkdir(parents=True,exist_ok=True)
creds,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive=build("drive","v3",credentials=creds,cache_discovery=False)
def entries(folder):
    data=[];token=None
    while True:
        resp=drive.files().list(q=f"'{folder}' in parents and trashed=false",
          pageSize=100,fields="nextPageToken,files(id,name,size,mimeType)",
          pageToken=token).execute(num_retries=4)
        data+=resp.get("files",[]);token=resp.get("nextPageToken")
        if not token:return data
source={f["name"]:f for f in entries(ROOT_ID)}
if ARCHIVE_NAME not in source:raise RuntimeError("Pachetul original 0001 lipsește din Drive")
existing=[f for f in entries(ROOT_ID) if f["name"]==OUT_NAME and f["mimeType"]=="application/vnd.google-apps.folder"]
DEST_ID=existing[0]["id"] if existing else drive.files().create(
   body={"name":OUT_NAME,"mimeType":"application/vnd.google-apps.folder","parents":[ROOT_ID]},
   fields="id").execute(num_retries=4)["id"]
def upload_json(name,object_):
    payload=json.dumps(object_,indent=2,ensure_ascii=False).encode("utf-8")
    media=MediaInMemoryUpload(payload,mimetype="application/json",resumable=False)
    prev=next((f for f in entries(DEST_ID) if f["name"]==name),None)
    if prev:drive.files().update(fileId=prev["id"],media_body=media,fields="id").execute(num_retries=4)
    else:drive.files().create(body={"name":name,"parents":[DEST_ID]},media_body=media,fields="id").execute(num_retries=4)
print("REA output folder: https://drive.google.com/drive/folders/"+DEST_ID)

In [ ]:
# Descărcare permisă a pachetului 0001; nu încerca dezactivarea protecției Google.
zip_path=WORK/ARCHIVE_NAME
if not zip_path.exists() or zip_path.stat().st_size!=int(source[ARCHIVE_NAME]["size"]):
    if zip_path.exists():zip_path.unlink()
    with zip_path.open("wb") as out:
        req=drive.files().get_media(fileId=source[ARCHIVE_NAME]["id"])
        downloader=MediaIoBaseDownload(out,req,chunksize=8*1024*1024)
        done=False
        while not done:
            _,done=downloader.next_chunk(num_retries=4)
    print("ZIP downloaded bytes:",zip_path.stat().st_size)
else:print("ZIP already downloaded.")
with zipfile.ZipFile(zip_path) as z:
    item=z.getinfo(EXE_MEMBER)
    if item.file_size!=18_102_272:raise RuntimeError("Unexpected executable size")
    exe=WORK/"ADBCD.exe"
    with z.open(item) as inp, exe.open("wb") as out:
        shutil.copyfileobj(inp,out,1024*1024)
hash_=hashlib.sha256(exe.read_bytes()).hexdigest()
if hash_!=EXE_SHA256:
    exe.unlink(missing_ok=True)
    raise RuntimeError("SOURCE_SHA256_MISMATCH: no further analysis")
print("AUTODATA_BINARY_INTEGRITY=PASS",hash_)
# Destroy raw ZIP copy to avoid retaining extra executable content in temporary Colab disk.
zip_path.unlink(missing_ok=True)

In [ ]:
# Instalează REA din pachetul oficial și versiunea Node compatibilă.
# În acest pas poate apărea output npm normal. Nu se execută Autodata.
TOOLS=Path("/content/autodata_rea_tooling")
TOOLS.mkdir(exist_ok=True)
REA_VERSION="6.8.0"
NODE_VERSION="24.22.1"
node=TOOLS/"node_modules"/"node"/"bin"/"node"
cli=TOOLS/"node_modules"/"rea-agents"/"scripts"/"rea.mjs"
if not node.is_file() or not cli.is_file():
    subprocess.run(["npm","install","--no-audit","--no-fund","--prefix",str(TOOLS),
       "node@"+NODE_VERSION,"rea-agents@"+REA_VERSION],check=True,timeout=600)
if not node.is_file() or not cli.is_file():
    raise RuntimeError("NODE_OR_REA_NOT_INSTALLED")
def run_rea(*args,timeout=240):
    cmd=[str(node),str(cli),*args]
    p=subprocess.run(cmd,cwd=str(WORK),text=True,capture_output=True,timeout=timeout)
    if p.returncode!=0:
        raise RuntimeError("REA command failed "+repr(args)+": "+p.stderr[-1200:])
    return p.stdout
version=run_rea("--version").strip()
print("REA_VERSION:",version)

In [ ]:
# Analyza REA original ADBCD.exe. Outputul rămâne numai în Drive privat.
status={"target":"original ADBCD.exe","target_sha256":EXE_SHA256,
        "rea_expected_version":REA_VERSION,"rea_reported_version":version,
        "executed_autodata":False,"license_bypass":False,
        "results":{}}
for report_name,args in [
    ("rea_pe_imports.json",["inspect-pe-imports",str(exe),"--json"]),
    ("rea_desktop_application.json",["inspect-desktop-application",str(WORK),"--json"])
]:
    try:
        raw=run_rea(*args)
        result=json.loads(raw)
        if not isinstance(result,dict):raise RuntimeError("Invalid REA JSON result")
        if result.get("ok") is False:raise RuntimeError("REA reported unsuccessful analysis")
        upload_json(report_name,result)
        status["results"][report_name]={"status":"PASS","json_bytes":len(raw.encode("utf-8"))}
        print("REA_STATIC_ANALYSIS=PASS",report_name,len(raw))
    except Exception as exc:
        status["results"][report_name]={"status":"FAIL","error":type(exc).__name__+": "+str(exc)[:1200]}
        print("REA_STATIC_ANALYSIS=FAIL",report_name,status["results"][report_name]["error"])
upload_json("REA_SUMMARY.json",status)
print("REA_RESULT_FOLDER=https://drive.google.com/drive/folders/"+DEST_ID)
if any(x["status"]!="PASS" for x in status["results"].values()):
    raise RuntimeError("One or more REA static inspections failed; check saved summary")